# Pixels-to-Predictions: Final Competition Notebook

This notebook is the **final, self-contained code** to reproduce our Kaggle submission for the ScienceQA-style visual MCQ challenge.

**Competition constraints (respected):**
- **Base model**: `HuggingFaceTB/SmolVLM-500M-Instruct`
- **No external data** (only competition CSV/images; optional self-generated captions are generated with the base model)
- **≤ 5M trainable parameters** for any fine-tuned weights

**Final inference pipeline (default in this notebook):**
- **Load base model + best legal LoRA adapter**: `data/adapters/mcq224_cont_letter`
- **Inference resolution**: **384×384**
- **Scoring**: deterministic **log-likelihood over letter completions** (`" A"`, `" B"`, …), pick argmax

**How to produce `submission.csv`:** run Sections **1 → 5** (Section 5 writes `submission.csv`).

---

In [22]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

In [23]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Kaggle zip extracts to: data/images/images/{train,val,test}
# and CSVs contain image_path like: images/train/train_00000.png
DATA_DIR = Path("data")
IMG_DIR = DATA_DIR / "images"

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Final competition settings (our best legal pipeline) ─────────────────────
# Inference image resolution used in our final full-val sweep.
IMG_SIZE = 384

# Load this adapter by default (best legal under 5M trainable params).
ADAPTER_DIR = DATA_DIR / "adapters" / "mcq224_cont_letter"
USE_ADAPTER = True

# Deterministic MCQ scoring (no sampling)
GEN_MAX_NEW_TOKENS = 12

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Inference IMG_SIZE: {IMG_SIZE}")
print(f"Adapter: {ADAPTER_DIR} (exists={ADAPTER_DIR.exists()})")

Using device: cuda
GPU: NVIDIA GeForce GTX 1080 Ti


## 2. Load and Preprocess Data

In [24]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


Note: we only drop rows missing **required** fields. `hint` / `lecture` may be NaN and are allowed.

In [25]:
# 2.1 missing data (drop only rows missing *required* fields)
# Note: `hint` / `lecture` can be NaN and should NOT cause rows to be dropped.

required_trainval = ["id", "image_path", "question", "choices", "num_choices", "answer"]
required_test = ["id", "image_path", "question", "choices", "num_choices"]

train_df = train_df.dropna(subset=required_trainval).reset_index(drop=True)
val_df   = val_df.dropna(subset=required_trainval).reset_index(drop=True)
test_df  = test_df.dropna(subset=required_test).reset_index(drop=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print("NaNs allowed in hint/lecture. Example counts:")
print("train hint NaN:", int(train_df["hint"].isna().sum()) if "hint" in train_df else "n/a")
print("train lecture NaN:", int(train_df["lecture"].isna().sum()) if "lecture" in train_df else "n/a")


Train: 3,109 | Val: 1,048 | Test: 1,008
NaNs allowed in hint/lecture. Example counts:
train hint NaN: 724
train lecture NaN: 440


In [26]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always successful

In [27]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(IMG_DIR / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=3109, val=1048, test=1008


## 3. Load model (+ adapter) and sanity-check generation

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and (by default) attaches the best legal adapter from `data/adapters/mcq224_cont_letter`.

The `generate()` output shown here is **only a smoke test**. The actual evaluation/submission uses **multiple-choice log-likelihood scoring** in Section 4.

In [28]:
# ── 3a. Load SmolVLM model (+ optional LoRA adapter) ────────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
if not torch.cuda.is_available():
    model.to(device)

if USE_ADAPTER:
    from peft import PeftModel

    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(
            f"Adapter dir not found: {ADAPTER_DIR}. "
            "Set ADAPTER_DIR to one of data/adapters/* (e.g., mcq224_cont_letter)."
        )
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)

model.eval()
print("Loaded model.")
if USE_ADAPTER:
    print("Loaded adapter:", ADAPTER_DIR.resolve())

# Quick generate() smoke test (NOT used for scoring/submission)
sample = val_df.iloc[0]
sample_image = Image.open(IMG_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(text=[sample_prompt], images=[sample_image], return_tensors="pt")
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:\n", sample_prompt)
print("\nModel output:\n", decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/home/fn2174/DL_final/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Prompt:
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always su

## 4. Multiple-Choice scoring (final decision)

`generate()` is not reliable for MCQ because it may output extra text.

Our final scoring policy is:
- For each option, score the **log-likelihood** of the completion being the **choice letter** (`" A"`, `" B"`, …)
- Pick the argmax score
- Output a **0-indexed integer** for Kaggle (`id,answer`)

This matches how we evaluated adapters on the full validation sweep.

In [29]:
# ── 4a. Log-likelihood scoring utilities (BATCHED, much faster) ───────────
# Final decision: score letter completions only (" A"/" B"/...), pick argmax.

@torch.inference_mode()
def predict_mcq_index(row: pd.Series, image: Image.Image) -> int:
    """Score each choice letter completion and return predicted 0-indexed answer."""
    prompt = build_prompt(row, include_answer=False)

    n = int(row["num_choices"]) if "num_choices" in row else len(row["choices"])
    n = min(n, len(CHOICE_LETTERS))

    completions = [f" {CHOICE_LETTERS[i]}" for i in range(n)]
    full_texts = [prompt + c for c in completions]

    inputs = processor(text=full_texts, images=[image] * n, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    prompt_ids = processor(text=[prompt], images=[image], return_tensors="pt")["input_ids"]
    prompt_len = int(prompt_ids.shape[1])

    logits = model(**inputs).logits  # (n, seq, vocab)
    input_ids = inputs["input_ids"]
    attn_mask = inputs.get("attention_mask", None)

    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = torch.log_softmax(shift_logits, dim=-1)
    token_logps = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)  # (n, seq-1)

    if attn_mask is not None:
        token_logps = token_logps * attn_mask[:, 1:].to(token_logps.dtype)

    start_pos = max(prompt_len - 1, 0)
    scores = []

    # Each completion is very short; computing the exact end position is cheap and avoids padding effects.
    for i in range(n):
        full_ids = processor(text=[full_texts[i]], images=[image], return_tensors="pt")["input_ids"]
        full_len = int(full_ids.shape[1])
        end_pos = max(full_len - 1, start_pos)
        slice_logps = token_logps[i, start_pos:end_pos]
        denom = slice_logps.numel() if slice_logps.numel() > 0 else 1
        scores.append(slice_logps.sum().item() / denom)

    return int(np.argmax(scores))

In [30]:
# ── Unified validation evaluation (used by 4b and Section 7) ───────────────
from tqdm.auto import tqdm


def evaluate_mcq_val(
    df: pd.DataFrame,
    n: int | None = None,
    *,
    start: int = 0,
    save_csv: Path | str | None = "auto",
    desc: str = "val acc",
    verbose: bool = True,
) -> pd.DataFrame:
    """Score `df` with `predict_mcq_index` and return per-row results.

    Columns: id, answer_true, answer_pred, correct, num_choices, image_path.

    ``save_csv``:
      - ``"auto"`` → ``data/val_eval_n{len}.csv``
      - a path → write there
      - ``None`` → do not write
    """
    if n is None:
        end = len(df)
    else:
        end = min(start + n, len(df))
    subset = df.iloc[start:end].reset_index(drop=True)

    chance = float((1.0 / subset["num_choices"]).mean()) if len(subset) else 0.0
    if verbose:
        print(
            f"Evaluating {len(subset)} val rows | "
            f"num_choices: {subset['num_choices'].value_counts().sort_index().to_dict()} | "
            f"~random baseline (mean 1/K): {chance:.4f}"
        )

    correct = 0
    first_wrong = None
    result_rows = []
    iterator = range(len(subset))
    if verbose:
        iterator = tqdm(iterator, desc=desc)

    for i in iterator:
        row = subset.iloc[i]
        img = Image.open(IMG_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        pred = predict_mcq_index(row, img)
        gold = int(row["answer"])
        ok = int(pred == gold)
        correct += ok
        if not ok and first_wrong is None:
            first_wrong = (i, row["id"], gold, pred, int(row["num_choices"]))

        result_rows.append(
            {
                "id": row["id"],
                "answer_true": gold,
                "answer_pred": int(pred),
                "correct": ok,
                "num_choices": int(row["num_choices"]),
                "image_path": row["image_path"],
            }
        )

    results = pd.DataFrame(result_rows)
    acc = correct / len(subset) if len(subset) else 0.0

    if verbose:
        print(f"Val accuracy: {acc:.4f}  (correct {correct}/{len(subset)})")

    if save_csv == "auto":
        out = DATA_DIR / f"val_eval_n{len(subset)}.csv"
        results.to_csv(out, index=False)
        if verbose:
            print(f"Saved per-row results to: {out.resolve()}")
    elif save_csv is not None:
        out = Path(save_csv)
        results.to_csv(out, index=False)
        if verbose:
            print(f"Saved per-row results to: {out.resolve()}")

    if verbose:
        if len(subset) and acc < chance * 1.05:
            print("Note: at or below ~random — expect this for a frozen baseline; LoRA fine-tuning usually helps.")
        if first_wrong is not None:
            j, eid, g, p, k = first_wrong
            print(f"First wrong (index {j}): id={eid} gold={g} pred={p} num_choices={k}")

    return results


In [31]:
import torch
print(torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

2.11.0+cu126
cuda_available: True
device: NVIDIA GeForce GTX 1080 Ti


In [32]:
# ── 4b. Sanity check on a validation slice ─────────────────────────────────
# Use 20 for a *quick* smoke test; 200–500 gives a much more stable accuracy estimate.
# Implementation: `evaluate_mcq_val` (cell above — shared with Section 7).
VAL_N = 20
evaluate_mcq_val(val_df, n=VAL_N)

Evaluating 20 val rows | num_choices: {3: 2, 4: 2, 5: 16} | ~random baseline (mean 1/K): 0.2183


val acc: 100%|██████████| 20/20 [06:29<00:00, 19.48s/it]

Val accuracy: 0.2000  (correct 4/20)
Saved per-row results to: /home/fn2174/DL_final/data/val_eval_n20.csv
Note: at or below ~random — expect this for a frozen baseline; LoRA fine-tuning usually helps.
First wrong (index 1): id=val_04111 gold=1 pred=0 num_choices=3


,id,answer_true,answer_pred,correct,num_choices,image_path
0,val_00671,0,0,1,3,images/val/val_00671.png
1,val_04111,1,0,0,3,images/val/val_04111.png
2,val_02022,3,0,0,4,images/val/val_02022.png
3,val_01237,0,4,0,5,images/val/val_01237.png
4,val_03458,4,4,1,5,images/val/val_03458.png
5,val_04064,3,3,1,4,images/val/val_04064.png
6,val_03959,4,0,0,5,images/val/val_03959.png
7,val_03289,2,4,0,5,images/val/val_03289.png
8,val_03452,3,4,0,5,images/val/val_03452.png
9,val_01021,0,0,1,5,images/val/val_01021.png


## 5. Create `submission.csv` (Kaggle-ready)

This cell runs inference for every row in `data/test.csv` and writes the Kaggle submission file.

- **Output format**: `id,answer` where `answer` is a **0-indexed integer**
- **Filename**: exactly `submission.csv`

After it finishes, upload the **root** `submission.csv` to Kaggle.

In [33]:
# ── 5a. Run test inference + write submission.csv ───────────────────────────
# This is the ONLY cell you need for Kaggle submission after running Sections 1–4.
from tqdm.auto import tqdm

pred_rows = []

for i in tqdm(range(len(test_df)), desc="predict test"):
    row = test_df.iloc[i]
    img = Image.open(IMG_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    pred_idx = predict_mcq_index(row, img)
    pred_rows.append({"id": row["id"], "answer": int(pred_idx)})

sub_df = pd.DataFrame(pred_rows)

# Kaggle expects EXACT filename: submission.csv
out_path_data = DATA_DIR / "submission.csv"
out_path_root = Path("submission.csv")
sub_df.to_csv(out_path_data, index=False)
sub_df.to_csv(out_path_root, index=False)

print(f"Wrote {len(sub_df)} rows to {out_path_root.resolve()}")
print(f"(also saved copy to {out_path_data.resolve()})")
print(sub_df.head())

# Quick format checks
assert list(sub_df.columns) == ["id", "answer"]
assert sub_df["answer"].dtype.kind in "iu"
assert sub_df["id"].is_unique
assert len(sub_df) == len(test_df)

## 7. Optional: evaluate on `val.csv`

This evaluates accuracy on `val.csv` using the **same scorer** as submission.

Notes:
- Run a small subset first (e.g. 200) to sanity-check speed.
- Full validation (1048) is slower but is the most reliable check.

In [34]:
# ── 7a. Validation evaluation (same helper as 4b) ──────────────────────────
# Quick sanity eval (increase to full val after speed looks OK)
VAL_EVAL_N = 200
evaluate_mcq_val(val_df, n=VAL_EVAL_N)

Evaluating 200 val rows | num_choices: {2: 58, 3: 67, 4: 32, 5: 43} | ~random baseline (mean 1/K): 0.3397


val acc: 100%|██████████| 200/200 [44:36<00:00, 13.38s/it] 

Val accuracy: 0.5350  (correct 107/200)
Saved per-row results to: /home/fn2174/DL_final/data/val_eval_n200.csv
First wrong (index 1): id=val_04111 gold=1 pred=0 num_choices=3


,id,answer_true,answer_pred,correct,num_choices,image_path
0,val_00671,0,0,1,3,images/val/val_00671.png
1,val_04111,1,0,0,3,images/val/val_04111.png
2,val_02022,3,0,0,4,images/val/val_02022.png
3,val_01237,0,4,0,5,images/val/val_01237.png
4,val_03458,4,4,1,5,images/val/val_03458.png
...,...,...,...,...,...,...
195,val_02814,1,2,0,3,images/val/val_02814.png
196,val_00844,1,2,0,3,images/val/val_00844.png
197,val_00634,2,2,1,3,images/val/val_00634.png
198,val_02290,1,2,0,3,images/val/val_02290.png


## 8. Optional: LoRA fine-tuning (≤ 5M trainable params)

This section shows how to train a small LoRA adapter **within the 5M trainable-parameter cap**.

- It is **OFF by default** (`RUN_SECTION8_TRAIN=False`) so you can safely run the notebook just to make a submission.
- If you enable training, it saves an adapter directory you can later load for inference.

In [ ]:
# ── 8a–8d. QLoRA fine-tuning (≤ 5M trainable params) ───────────────────────
# This block trains a small LoRA adapter and keeps the base model frozen.

from dataclasses import dataclass
from typing import Optional

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


def count_trainable_params(m: torch.nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def infer_num_layers(m) -> Optional[int]:
    """Resolve LLM depth for VLM stacks (e.g. SmolVLM / Idefics3: `config.text_config`)."""
    keys = ("num_hidden_layers", "n_layer", "num_layers")

    def _from_cfg(cfg) -> Optional[int]:
        if cfg is None:
            return None
        for k in keys:
            v = getattr(cfg, k, None)
            if isinstance(v, int):
                return v
        return None

    root = getattr(m, "config", None)
    if root is not None:
        for nested in ("text_config", "language_config", "llm_config"):
            v = _from_cfg(getattr(root, nested, None))
            if v is not None:
                return v
        v = _from_cfg(root)
        if v is not None:
            return v

    for attr in ("text_config", "language_config"):
        v = _from_cfg(getattr(m, attr, None))
        if v is not None:
            return v

    lm = getattr(m, "language_model", None)
    if lm is not None and hasattr(lm, "config"):
        v = _from_cfg(lm.config)
        if v is not None:
            return v
    return None


# 8a) Load processor + 4-bit model
processor_ft = AutoProcessor.from_pretrained(MODEL_ID)
if processor_ft.tokenizer.pad_token is None:
    processor_ft.tokenizer.pad_token = processor_ft.tokenizer.eos_token

# Finetuning-only (Section 8): keep vision/text activations small so 11GB GPUs survive Trainer.
# Inference elsewhere can still use IMG_SIZE=224.
FT_IMG_SIZE = 192

import gc

gc.collect()
torch.cuda.empty_cache()

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

ft_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_cfg,
)
ft_model = prepare_model_for_kbit_training(ft_model)

# 8b) LoRA config (keep under 5M params)
num_layers = infer_num_layers(ft_model)
last_k = 8
layers_to_transform = None
if isinstance(num_layers, int) and num_layers > last_k:
    layers_to_transform = list(range(num_layers - last_k, num_layers))
elif isinstance(num_layers, int):
    layers_to_transform = list(range(num_layers))

# Prefer the auto-picked config from Section 9b (maximizes params under the 5M cap).
if "LORA_CFG_UNDER_5M" in globals():
    lora_cfg = LORA_CFG_UNDER_5M
    print(
        "Using LORA_CFG_UNDER_5M from Section 9b: "
        f"r={lora_cfg.r} targets={len(lora_cfg.target_modules)} layers={len(lora_cfg.layers_to_transform)}"
    )
else:
    # Fallback: small attention-only LoRA.
    # PEFT requires `layers_to_transform` whenever `layers_pattern` is set.
    _lora_kw = dict(
        r=4,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
    )
    if layers_to_transform is not None:
        _lora_kw["layers_to_transform"] = layers_to_transform
        _lora_kw["layers_pattern"] = "layers"

    lora_cfg = LoraConfig(**_lora_kw)
    if num_layers is None:
        print(
            "Warning: could not infer num_hidden_layers; LoRA applies to all matched layers. "
            "Watch trainable param count vs 5M cap."
        )

ft_model = get_peft_model(ft_model, lora_cfg)

# Trainer skips nn.DataParallel only when `is_loaded_in_8bit` is set. 4-bit QLoRA sets
# `is_loaded_in_4bit` instead, so with 2+ visible GPUs Trainer would still wrap the model
# in DataParallel — that breaks bitsandbytes `device_map="auto"` and often OOMs.
ft_model.is_loaded_in_8bit = True

_base = ft_model.get_base_model()
if hasattr(_base, "gradient_checkpointing_enable"):
    try:
        _base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        _base.gradient_checkpointing_enable()
    print("Gradient checkpointing: on (base model)")

ft_model.print_trainable_parameters()
trainable = count_trainable_params(ft_model)
print("Trainable params:", trainable)
assert trainable <= 5_000_000, f"Trainable params {trainable} exceeds 5M cap"


# 8c) Dataset that trains only on the answer token(s)
# IMPORTANT: do NOT use processor truncation with Idefics3/SmolVLM.
# The processor expands `<image>` into many internal tokens; truncation can cut that expansion and
# triggers `Mismatch in image token count ... Likely due to truncation='max_length'`.
# Instead, we keep a SHORT finetuning prompt so inputs naturally fit.

def build_prompt_ft(row: pd.Series) -> str:
    """Short prompt for finetuning (avoid long lecture/hint causing OOM)."""
    n = int(row["num_choices"]) if "num_choices" in row else len(row["choices"])
    n = min(n, len(CHOICE_LETTERS))

    q = str(row["question"]).strip()
    choices = row["choices"]
    if not isinstance(choices, (list, tuple)):
        choices = list(choices)

    lines = [
        "<image>",
        "You are given a multiple-choice question.",
        f"Question: {q}",
        "Choices:",
    ]

    # Clip per-choice text to keep sequences bounded.
    def _clip(s: str, max_chars: int = 160) -> str:
        s = (s or "").strip().replace("\n", " ")
        return s if len(s) <= max_chars else (s[: max_chars - 1] + "…")

    for i in range(n):
        lines.append(f"{CHOICE_LETTERS[i]}. {_clip(str(choices[i]))}")

    lines.append("Answer:")
    return "\n".join(lines) + "\n"


class LoRAMCQDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(IMG_DIR / row["image_path"]).convert("RGB").resize((FT_IMG_SIZE, FT_IMG_SIZE))

        prompt = build_prompt_ft(row)
        ans_idx = int(row["answer"])
        ans_letter = CHOICE_LETTERS[ans_idx]
        completion = f" {ans_letter}"
        full_text = prompt + completion

        n_completion = len(processor_ft.tokenizer.encode(completion, add_special_tokens=False))

        enc = processor_ft(text=full_text, images=img, return_tensors="pt")
        enc = {k: v.squeeze(0) for k, v in enc.items()}

        labels = enc["input_ids"].clone()
        seq_len = int(labels.shape[0])
        n_sup = min(n_completion, seq_len)
        labels[:] = -100
        labels[-n_sup:] = enc["input_ids"][-n_sup:]

        if "attention_mask" in enc:
            labels[enc["attention_mask"] == 0] = -100

        if "pixel_values" in enc and enc["pixel_values"].dtype == torch.float32:
            enc["pixel_values"] = enc["pixel_values"].to(torch.float16)

        enc["labels"] = labels
        return enc


train_small = train_df.sample(n=min(3000, len(train_df)), random_state=SEED).reset_index(drop=True)
val_small = val_df.sample(n=min(500, len(val_df)), random_state=SEED).reset_index(drop=True)

train_ds_ft = LoRAMCQDataset(train_small)
val_ds_ft = LoRAMCQDataset(val_small)

@dataclass
class Collator:
    pad_to_multiple_of: int = 8

    def __call__(self, features):
        batch = {}
        pad_id = processor_ft.tokenizer.pad_token_id
        if pad_id is None:
            pad_id = int(processor_ft.tokenizer.eos_token_id)

        def _1d_long(x: torch.Tensor) -> torch.Tensor:
            x = x.squeeze()
            if x.dim() != 1:
                raise ValueError(f"expected 1D token tensor, got shape {tuple(x.shape)}")
            return x.long()

        input_ids = [_1d_long(f["input_ids"]) for f in features]
        attn = [_1d_long(f["attention_mask"]) for f in features]
        labels = [_1d_long(f["labels"]) for f in features]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=pad_id)
        attention_mask = pad_sequence(attn, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels = labels.masked_fill(attention_mask == 0, -100)

        if self.pad_to_multiple_of:
            m = input_ids.shape[1] % self.pad_to_multiple_of
            if m:
                extra = self.pad_to_multiple_of - m
                input_ids = torch.nn.functional.pad(input_ids, (0, extra), value=pad_id)
                attention_mask = torch.nn.functional.pad(attention_mask, (0, extra), value=0)
                labels = torch.nn.functional.pad(labels, (0, extra), value=-100)

        batch["input_ids"] = input_ids
        batch["attention_mask"] = attention_mask
        batch["labels"] = labels

        if features and "pixel_values" in features[0]:
            pv = torch.stack([f["pixel_values"].squeeze(0) if f["pixel_values"].dim() == 4 else f["pixel_values"] for f in features])
            batch["pixel_values"] = pv.to(dtype=torch.float16) if pv.dtype == torch.float32 else pv

        return batch


# 8d) Train
args = TrainingArguments(
    output_dir=str(DATA_DIR / "lora_out"),
    # Batch 1 is safer on 11GB cards with long multimodal sequences; effective batch via grad_accum.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    bf16=False,
    fp16=True,
    report_to=[],
)

def train_lora_adapter(
    lora_cfg: "LoraConfig",
    *,
    adapter_out_dir: Path,
    run_name: str,
    train_n: int = 3000,
    val_n: int = 500,
    num_train_epochs: float = 1.0,
    make_submission: bool = True,
) -> Path:
    """Self-contained PEFT training run (safe for Run-All).

    What gets saved locally:
    - Trainer checkpoints under `data/runs/<run_name>/checkpoints/` (no limit)
    - Final adapter weights under `adapter_out_dir/`
    - `run_meta.json` under `adapter_out_dir/`
    - `data/submission_<run_name>.csv` (if `make_submission=True`)
    """
    from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig, TrainingArguments, Trainer
    from peft import get_peft_model, prepare_model_for_kbit_training

    proc = AutoProcessor.from_pretrained(MODEL_ID)
    if proc.tokenizer.pad_token is None:
        proc.tokenizer.pad_token = proc.tokenizer.eos_token

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    m = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=bnb,
        low_cpu_mem_usage=True,
    )
    m = prepare_model_for_kbit_training(m)
    m = get_peft_model(m, lora_cfg)

    # Prevent Trainer from wrapping into nn.DataParallel when multiple GPUs are visible.
    m.is_loaded_in_8bit = True

    base = m.get_base_model()
    if hasattr(base, "gradient_checkpointing_enable"):
        try:
            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            base.gradient_checkpointing_enable()

    m.print_trainable_parameters()
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    assert trainable <= 5_000_000, f"Trainable params {trainable} exceeds 5M cap"

    # Use the short finetuning prompt + dataset/collator defined above.
    global processor_ft
    processor_ft = proc

    train_small = train_df.sample(n=min(train_n, len(train_df)), random_state=SEED).reset_index(drop=True)
    val_small = val_df.sample(n=min(val_n, len(val_df)), random_state=SEED).reset_index(drop=True)

    train_ds = LoRAMCQDataset(train_small)
    val_ds = LoRAMCQDataset(val_small)

    run_dir = DATA_DIR / "runs" / run_name
    ckpt_dir = run_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    targs = TrainingArguments(
        output_dir=str(ckpt_dir),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        gradient_checkpointing=True,
        learning_rate=2e-4,
        num_train_epochs=num_train_epochs,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=None,
        bf16=False,
        fp16=True,
        report_to=[],
    )

    tr = Trainer(
        model=m,
        args=targs,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=Collator(),
    )

    train_result = tr.train()

    adapter_out_dir.mkdir(parents=True, exist_ok=True)
    m.save_pretrained(adapter_out_dir)

    meta = {
        "run_name": run_name,
        "model_id": MODEL_ID,
        "seed": int(SEED),
        "ft_img_size": int(FT_IMG_SIZE),
        "train_n": int(min(train_n, len(train_df))),
        "val_n": int(min(val_n, len(val_df))),
        "num_train_epochs": float(num_train_epochs),
        "trainable_params": int(trainable),
        "checkpoint_dir": str(ckpt_dir),
        "train": {
            "global_step": int(getattr(tr.state, "global_step", 0) or 0),
            "train_loss": float(getattr(train_result, "training_loss", float("nan"))),
        },
        "lora": {
            "r": int(lora_cfg.r),
            "lora_alpha": int(getattr(lora_cfg, "lora_alpha", 0) or 0),
            "use_dora": bool(getattr(lora_cfg, "use_dora", False)),
            "target_modules": list(lora_cfg.target_modules) if getattr(lora_cfg, "target_modules", None) is not None else None,
            "layers_to_transform": list(lora_cfg.layers_to_transform) if getattr(lora_cfg, "layers_to_transform", None) is not None else None,
            "layers_pattern": getattr(lora_cfg, "layers_pattern", None),
        },
    }
    (adapter_out_dir / "run_meta.json").write_text(json.dumps(meta, indent=2))

    sub_path = None
    if make_submission:
        # Uses the existing scorer (`predict_mcq_index`) and writes a distinct file.
        # Requires Sections 1–5 already ran (defines `test_df`, `IMG_DIR`, `model`, `processor`, `predict_mcq_index`).
        try:
            from peft import PeftModel

            adapter_model = PeftModel.from_pretrained(model, adapter_out_dir)
            pred_rows = []
            for i in tqdm(range(len(test_df)), desc=f"test submit: {run_name}"):
                row = test_df.iloc[i]
                img = Image.open(IMG_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
                pred_idx = predict_mcq_index(row, img)
                pred_rows.append({"id": row["id"], "answer": int(pred_idx)})

            sub_df = pd.DataFrame(pred_rows)
            sub_path = DATA_DIR / f"submission_{run_name}.csv"
            sub_df.to_csv(sub_path, index=False)
            meta["submission_csv"] = str(sub_path)
            (adapter_out_dir / "run_meta.json").write_text(json.dumps(meta, indent=2))
            print("Saved submission to", sub_path.resolve())
        except Exception as e:
            print("Warning: could not write submission for run", run_name, "error:", repr(e))

    print("Saved adapter to", adapter_out_dir.resolve())
    print("Saved run metadata to", (adapter_out_dir / "run_meta.json").resolve())
    if sub_path is not None:
        print("Saved submission to", sub_path.resolve())
    print("Saved checkpoints under", ckpt_dir.resolve())
    return adapter_out_dir


# Section 8 default: do NOT auto-train during Run-All.
RUN_SECTION8_TRAIN = False
if RUN_SECTION8_TRAIN:
    train_lora_adapter(lora_cfg, adapter_out_dir=DATA_DIR / "lora_adapter", run_name="section8")
else:
    print("Section 8: training skipped (RUN_SECTION8_TRAIN=False).")

# After training, load adapter (Section 8e) and evaluate/submit with it.

/home/fn2174/DL_final/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Using LORA_CFG_UNDER_5M from Section 9b: r=16 targets=7 layers=16
Gradient checkpointing: on (base model)
trainable params: 4,341,760 || all params: 511,824,064 || trainable%: 0.8483
Trainable params: 4341760
Section 8: training skipped (RUN_SECTION8_TRAIN=False).


In [36]:
# ── 8e. Load LoRA adapter for inference (optional) ─────────────────────────-
# If you trained and saved an adapter, you can load it into the base model.

from peft import PeftModel

adapter_path = DATA_DIR / "lora_adapter"
if adapter_path.exists():
    # `model` is the inference model used by predict_mcq_index above.
    # Attach adapter weights on top of the existing base model.
    model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    print("Loaded adapter into inference model:", adapter_path.resolve())
else:
    print("No adapter found at", adapter_path.resolve())

No adapter found at /home/fn2174/DL_final/data/lora_adapter


## 9. Expand LoRA targets to MLP layers 
Currently LoRA only adapts attention projections (q/k/v/o). Adding MLP layers (gate/up/down_proj) lets the model adapt its internal representations, not just how it attends. Research shows MLP-only LoRA can outperform attention-only even at higher rank. Reduce rank to fit the 5M param budget.

In [37]:
# ── 9. Expand LoRA targets to include MLP projections ───────────────────────
# Goal: adapt feed-forward blocks (gate/up/down_proj), not only attention.
# Prereq: Section 1 (paths, MODEL_ID) and GPU. Loads a temporary 4-bit model to COUNT trainable params.

from transformers import AutoConfig, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


def _text_num_hidden_layers() -> int:
    cfg = AutoConfig.from_pretrained(MODEL_ID)
    tc = getattr(cfg, "text_config", None)
    if tc is not None and getattr(tc, "num_hidden_layers", None) is not None:
        return int(tc.num_hidden_layers)
    raise ValueError("Could not read text_config.num_hidden_layers from config")


def _last_k_layer_indices(last_k: int) -> list[int]:
    n = _text_num_hidden_layers()
    last_k = min(last_k, n)
    return list(range(n - last_k, n))


def count_trainable_lora_params(lora_cfg: LoraConfig) -> int:
    """Build a fresh 4-bit model, attach LoRA, count trainable params, then free VRAM."""
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    m = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        quantization_config=bnb,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    m = prepare_model_for_kbit_training(m)
    m = get_peft_model(m, lora_cfg)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    del m
    torch.cuda.empty_cache()
    return int(n)


# Attention + MLP (Llama-style blocks use gate_proj / up_proj / down_proj)
LORA_CFG_SECTION9 = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    layers_to_transform=_last_k_layer_indices(6),
    layers_pattern="layers",
    task_type="CAUSAL_LM",
)

n9 = count_trainable_lora_params(LORA_CFG_SECTION9)
print("Section 9 — trainable LoRA params:", f"{n9:,}")
assert n9 <= 5_000_000, f"Exceeds 5M cap: {n9:,}. Lower r or last_k in this cell."

S9_ADAPTER_DIR = DATA_DIR / "lora_s9_adapter"

RUN_SECTION9_TRAIN = False
if RUN_SECTION9_TRAIN:
    if "train_lora_adapter" not in globals():
        raise RuntimeError("Run Section 8 first (defines `train_lora_adapter`).")
    train_lora_adapter(LORA_CFG_SECTION9, adapter_out_dir=S9_ADAPTER_DIR, run_name="section9_lora")
else:
    print(
        "\nTo train: set `RUN_SECTION9_TRAIN=True` in this cell (Section 9). "
        f"Adapter will save to: {S9_ADAPTER_DIR}"
    )


/home/fn2174/DL_final/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Section 9 — trainable LoRA params: 330,240

To train: set `RUN_SECTION9_TRAIN=True` in this cell (Section 9). Adapter will save to: data/lora_s9_adapter


## 9b. Auto-pick a LoRA config **under 5M trainable params**

This helper searches a small grid of LoRA settings and picks the **largest** configuration that stays within the competition cap (**≤ 5,000,000 trainable parameters**).

- We count **only trainable parameters** (LoRA adapter weights). Base model weights remain frozen.
- This section uses `count_trainable_lora_params(...)` from Section 9 (it temporarily builds a fresh 4-bit model to measure params).


In [38]:
# Prereq: run Section 9 first (defines `_last_k_layer_indices` and `count_trainable_lora_params`).
if "count_trainable_lora_params" not in globals() or "_last_k_layer_indices" not in globals():
    raise RuntimeError("Run Section 9 first.")

from peft import LoraConfig

CAP_TRAINABLE = 5_000_000

# Candidate module sets (increasing adaptation capacity)
TARGET_SETS = [
    ["q_proj", "v_proj"],
    ["q_proj", "k_proj", "v_proj", "o_proj"],
    ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
]

# Small search grid (kept small to avoid wasting time)
R_VALUES = [2, 4, 8, 16]
LAST_K_VALUES = [2, 4, 6, 8, 12, 16]


def find_best_lora_cfg_under_cap(cap: int = CAP_TRAINABLE):
    best = None  # (trainable, cfg)

    for targets in TARGET_SETS:
        for r in R_VALUES:
            for last_k in LAST_K_VALUES:
                cfg = LoraConfig(
                    r=r,
                    lora_alpha=max(8, 4 * r),
                    lora_dropout=0.05,
                    bias="none",
                    target_modules=targets,
                    layers_to_transform=_last_k_layer_indices(last_k),
                    layers_pattern="layers",
                    task_type="CAUSAL_LM",
                )

                n = count_trainable_lora_params(cfg)
                ok = n <= cap
                print(
                    f"targets={len(targets):2d} r={r:2d} last_k={last_k:2d} -> trainable={n:,} "
                    + ("OK" if ok else "OVER")
                )

                if ok and (best is None or n > best[0]):
                    best = (n, cfg)

    return best


best = find_best_lora_cfg_under_cap(CAP_TRAINABLE)
if best is None:
    raise RuntimeError("No config in the search grid fit under the 5M cap. Reduce targets/r/last_k.")

best_n, best_cfg = best
print("\nBest under cap:")
print("trainable params:", f"{best_n:,}")
print("target_modules:", best_cfg.target_modules)
print("r:", best_cfg.r)
print("layers_to_transform (count):", len(best_cfg.layers_to_transform))

# Use this in training by setting your LoRA config to `best_cfg`.
LORA_CFG_UNDER_5M = best_cfg


targets= 2 r= 2 last_k= 2 -> trainable=12,800 OK


/home/fn2174/DL_final/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


targets= 2 r= 2 last_k= 4 -> trainable=25,600 OK
targets= 2 r= 2 last_k= 6 -> trainable=38,400 OK
targets= 2 r= 2 last_k= 8 -> trainable=51,200 OK
targets= 2 r= 2 last_k=12 -> trainable=76,800 OK
targets= 2 r= 2 last_k=16 -> trainable=102,400 OK
targets= 2 r= 4 last_k= 2 -> trainable=25,600 OK
targets= 2 r= 4 last_k= 4 -> trainable=51,200 OK


/home/fn2174/DL_final/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


targets= 2 r= 4 last_k= 6 -> trainable=76,800 OK


...

targets= 7 r= 8 last_k= 4 -> trainable=542,720 OK
targets= 7 r= 8 last_k= 6 -> trainable=814,080 OK


/home/fn2174/DL_final/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


targets= 7 r= 8 last_k= 8 -> trainable=1,085,440 OK
targets= 7 r= 8 last_k=12 -> trainable=1,628,160 OK
targets= 7 r= 8 last_k=16 -> trainable=2,170,880 OK
targets= 7 r=16 last_k= 2 -> trainable=542,720 OK
targets= 7 r=16 last_k= 4 -> trainable=1,085,440 OK


/home/fn2174/DL_final/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


targets= 7 r=16 last_k= 6 -> trainable=1,628,160 OK
targets= 7 r=16 last_k= 8 -> trainable=2,170,880 OK
targets= 7 r=16 last_k=12 -> trainable=3,256,320 OK
targets= 7 r=16 last_k=16 -> trainable=4,341,760 OK

Best under cap:
trainable params: 4,341,760
target_modules: {'o_proj', 'gate_proj', 'down_proj', 'q_proj', 'up_proj', 'k_proj', 'v_proj'}
r: 16
layers_to_transform (count): 16


## 10. Upgrade LoRA to DoRA 
DoRA decomposes weight updates into magnitude and direction components, giving better fine-tuning quality than standard LoRA with zero extra inference cost. A config-level change with documented gains on VLM benchmarks.

In [ ]:
# ── 10. DoRA (direction-only low-rank adaptation) ─────────────────────────
# Same targets as Section 9, but `use_dora=True` in PEFT's LoraConfig.
# Trainable count is slightly higher than plain LoRA; this cell checks the 5M budget.
# Prereq: run Section 9 first (defines `_last_k_layer_indices` and `count_trainable_lora_params`).

if "count_trainable_lora_params" not in globals() or "_last_k_layer_indices" not in globals():
    raise RuntimeError("Run the Section 9 code cell first.")

from peft import LoraConfig

LORA_CFG_SECTION10 = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_dora=True,
    target_modules=["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    layers_to_transform=_last_k_layer_indices(6),
    layers_pattern="layers",
    task_type="CAUSAL_LM",
)

n10 = count_trainable_lora_params(LORA_CFG_SECTION10)
print("Section 10 (DoRA) — trainable params:", f"{n10:,}")
assert n10 <= 5_000_000, f"Exceeds 5M cap: {n10:,}. Try r=2 or last_k=4."

DORA_ADAPTER_DIR = DATA_DIR / "dora_adapter"

# Run-All friendly: this section can train by itself.
RUN_SECTION10_TRAIN = True
if RUN_SECTION10_TRAIN:
    if "train_lora_adapter" not in globals():
        raise RuntimeError("Run Section 8 first (defines `train_lora_adapter`).")

    train_lora_adapter(LORA_CFG_SECTION10, adapter_out_dir=DORA_ADAPTER_DIR, run_name="section10_dora")
else:
    print(
        "\nTo train: set `RUN_SECTION10_TRAIN=True` in this cell (Section 10)."
    )


## 11. Add self-generated image captions to prompts 
Use the model itself to describe each image, then feed that description back as extra text context during training and inference. Small VLMs struggle with raw visual extraction; text bridges the gap.

In [ ]:
# ── 11. Self-generated image captions (same model, no external data) ───────
# Uses SmolVLM once per image to produce a short description, then prepends it to the MCQ prompt.
# Prereq: Section 3 loaded `model`, `processor`, and Section 2 defined `build_prompt`, `CHOICE_LETTERS`.

CAPTION_MAX_NEW = 80
_CAPTION_CACHE: dict[str, str] = {}


def inject_caption_into_prompt(base_prompt: str, caption: str) -> str:
    """Insert caption right after the leading `<image>` line."""
    caption = (caption or "").strip()
    if not caption:
        return base_prompt
    if not base_prompt.lstrip().startswith("<image>"):
        return f"Image description:\n{caption}\n\n" + base_prompt
    first_nl = base_prompt.find("\n")
    rest = base_prompt[first_nl + 1 :] if first_nl != -1 else ""
    return f"<image>\nImage description:\n{caption}\n\n" + rest.lstrip()


@torch.inference_mode()
def generate_image_caption(img: Image.Image, use_cache_key: str | None = None) -> str:
    if use_cache_key and use_cache_key in _CAPTION_CACHE:
        return _CAPTION_CACHE[use_cache_key]

    cap_prompt = (
        "<image>\n"
        "Briefly describe this image for a student answering a science multiple-choice question. "
        "Mention any visible text, axes, units, maps, or diagrams. One or two short sentences only.\n"
        "Description:"
    )
    inputs = processor(text=[cap_prompt], images=[img], return_tensors="pt")
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=CAPTION_MAX_NEW, do_sample=False)
    text = processor.batch_decode(out, skip_special_tokens=True)[0]
    # Keep only text after "Description:" if the model echoed the prompt
    if "Description:" in text:
        text = text.rsplit("Description:", 1)[-1]
    cap = text.strip()[:500]
    if use_cache_key:
        _CAPTION_CACHE[use_cache_key] = cap
    return cap


@torch.inference_mode()
def predict_mcq_index_captioned(row: pd.Series, image: Image.Image) -> int:
    """Same batched log-likelihood scorer as Section 4, but with an injected image caption."""
    cap = generate_image_caption(image, use_cache_key=str(row.get("image_path", "")))
    prompt = inject_caption_into_prompt(build_prompt(row, include_answer=False), cap)

    n = int(row["num_choices"]) if "num_choices" in row else len(row["choices"])
    n = min(n, len(CHOICE_LETTERS))

    completions = [f" {CHOICE_LETTERS[i]}" for i in range(n)]
    full_texts = [prompt + c for c in completions]
    images = [image] * n

    inputs = processor(text=full_texts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    inp_prompt = processor(text=prompt, images=image, return_tensors="pt")
    prompt_len = int(inp_prompt["input_ids"].shape[1])

    outputs = model(**inputs)
    logits = outputs.logits
    input_ids = inputs["input_ids"]
    attn_mask = inputs.get("attention_mask", None)

    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = torch.log_softmax(shift_logits, dim=-1)
    token_logps = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)

    if attn_mask is not None:
        shift_mask = attn_mask[:, 1:].to(token_logps.dtype)
        token_logps = token_logps * shift_mask

    start_pos = max(prompt_len - 1, 0)
    scores = []
    for i in range(n):
        inp_full = processor(text=full_texts[i], images=image, return_tensors="pt")
        full_len = int(inp_full["input_ids"].shape[1])
        if full_len <= prompt_len:
            scores.append(float("-inf"))
            continue
        end_pos = full_len - 1
        slice_logps = token_logps[i, start_pos:end_pos]
        denom = max(slice_logps.numel(), 1)
        scores.append(slice_logps.sum().item() / denom)

    return int(np.argmax(scores))


# Demo on one validation row (optional)
_demo = val_df.iloc[0]
_demo_img = Image.open(IMG_DIR / _demo["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
print("Caption:", generate_image_caption(_demo_img, use_cache_key=str(_demo["image_path"])))
print("Pred (no caption):", predict_mcq_index(_demo, _demo_img))
print("Pred (captioned): ", predict_mcq_index_captioned(_demo, _demo_img), "| gold:", int(_demo["answer"]))



## 12. Try targeting both attention and MLP layers (q_proj, v_proj, gate_proj, up_proj, down_proj) at a lower rank, while keeping param count under 5M.

In [ ]:
# ── 12. Attention + MLP at lower rank (stay under 5M trainable) ─────────────
# Prereq: Section 9 cell (defines helpers and `count_trainable_lora_params`).

if "count_trainable_lora_params" not in globals() or "_last_k_layer_indices" not in globals():
    raise RuntimeError("Run the Section 9 code cell first.")

from peft import LoraConfig

# Lower rank keeps the budget safe when targeting many modules.
LORA_CFG_SECTION12 = LoraConfig(
    r=2,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    layers_to_transform=_last_k_layer_indices(6),
    layers_pattern="layers",
    task_type="CAUSAL_LM",
)

n12 = count_trainable_lora_params(LORA_CFG_SECTION12)
print("Section 12 — trainable params:", f"{n12:,}")
assert n12 <= 5_000_000, f"Exceeds 5M cap: {n12:,}. Reduce r to 1 or last_k to 4."

S12_ADAPTER_DIR = DATA_DIR / "lora_s12_adapter"

RUN_SECTION12_TRAIN = False
if RUN_SECTION12_TRAIN:
    if "train_lora_adapter" not in globals():
        raise RuntimeError("Run Section 8 first (defines `train_lora_adapter`).")
    train_lora_adapter(LORA_CFG_SECTION12, adapter_out_dir=S12_ADAPTER_DIR, run_name="section12_lora")
else:
    print(
        "\nTo train: set `RUN_SECTION12_TRAIN=True` in this cell (Section 12). "
        f"Adapter will save to: {S12_ADAPTER_DIR}"
    )

# Optional: compare budgets if you already ran Sections 9–10 in this session
for name in ("n9", "n10", "n12"):
    if name in globals():
        print(f"  {name} = {globals()[name]:,} trainable params")


Section 12 — trainable params: 165,120

To train: set `lora_cfg = LORA_CFG_SECTION12` in Section 8 and re-run fine-tuning. Compare validation accuracy vs Section 9 (r=4) and Section 10 (DoRA).
  n9 = 330,240 trainable params
  n10 = 374,400 trainable params
  n12 = 165,120 trainable params


## 13. Experiment with changes like "The correct answer is:" vs "Answer:" or reordering context vs question.

In [ ]:
# ── 13. Prompt variants (answer cue + context order) ────────────────────────
# Prereq: Section 2 (`CHOICE_LETTERS`), same CSV columns as `build_prompt`.


def build_prompt_variant(
    row: pd.Series,
    *,
    include_answer: bool = False,
    answer_prefix: str = "Answer:",
    context_before_question: bool = True,
) -> str:
    """Same structure as `build_prompt`, but you can change the final cue and order."""
    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    question_block = f"Question: {row['question']}\nChoices:\n{choices_str}\n"

    prompt = "<image>\n"
    if context_before_question:
        if context_str:
            prompt += f"Context:\n{context_str}\n\n"
        prompt += question_block
    else:
        prompt += question_block
        if context_str:
            prompt += f"\nContext:\n{context_str}\n"

    # Final answer cue (completion during scoring is " A", " B", ...)
    ap = answer_prefix.rstrip()
    prompt += ap

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt


@torch.inference_mode()
def predict_mcq_index_with_prompt_builder(row: pd.Series, image: Image.Image, prompt_builder) -> int:
    """Batched log-likelihood scorer using a custom prompt function(row, include_answer=False)."""
    prompt = prompt_builder(row, include_answer=False)

    n = int(row["num_choices"]) if "num_choices" in row else len(row["choices"])
    n = min(n, len(CHOICE_LETTERS))

    completions = [f" {CHOICE_LETTERS[i]}" for i in range(n)]
    full_texts = [prompt + c for c in completions]
    images = [image] * n

    inputs = processor(text=full_texts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    inp_prompt = processor(text=prompt, images=image, return_tensors="pt")
    prompt_len = int(inp_prompt["input_ids"].shape[1])

    outputs = model(**inputs)
    logits = outputs.logits
    input_ids = inputs["input_ids"]
    attn_mask = inputs.get("attention_mask", None)

    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = torch.log_softmax(shift_logits, dim=-1)
    token_logps = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)

    if attn_mask is not None:
        token_logps = token_logps * attn_mask[:, 1:].to(token_logps.dtype)

    start_pos = max(prompt_len - 1, 0)
    scores = []
    for i in range(n):
        inp_full = processor(text=full_texts[i], images=image, return_tensors="pt")
        full_len = int(inp_full["input_ids"].shape[1])
        if full_len <= prompt_len:
            scores.append(float("-inf"))
            continue
        end_pos = full_len - 1
        slice_logps = token_logps[i, start_pos:end_pos]
        scores.append(slice_logps.sum().item() / max(slice_logps.numel(), 1))

    return int(np.argmax(scores))


# Quick A/B on one validation example (swap prefix / order and compare argmax)
_row = val_df.iloc[0]
_img = Image.open(IMG_DIR / _row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))

variants = [
    ("Answer:", True),
    ("The correct answer is:", True),
    ("Answer:", False),  # context after question
]

for prefix, ctx_first in variants:
    pred = predict_mcq_index_with_prompt_builder(
        _row,
        _img,
        lambda r, include_answer=False: build_prompt_variant(
            r,
            include_answer=include_answer,
            answer_prefix=prefix,
            context_before_question=ctx_first,
        ),
    )
    print(f"prefix={prefix!r} context_first={ctx_first} -> pred={pred} gold={int(_row['answer'])}")


## 14. You have metadata columns like subject, grade, and topic. Try adding one of them to your prompt and see if it helps the model reason better.

In [ ]:
# ── 14. Add metadata (subject, grade, topic) to the prompt ─────────────────
# Prereq: Section 2 (`CHOICE_LETTERS`) and Section 13 (`predict_mcq_index_with_prompt_builder`).

if "predict_mcq_index_with_prompt_builder" not in globals():
    raise RuntimeError("Run Section 13 first (defines predict_mcq_index_with_prompt_builder).")


def build_prompt_with_metadata(
    row: pd.Series,
    include_answer: bool = False,
    *,
    fields: tuple[str, ...] = ("grade", "subject", "topic"),
) -> str:
    """Like `build_prompt`, with an extra header built from metadata columns."""
    meta_lines = []
    for f in fields:
        if f in row.index and pd.notna(row[f]) and str(row[f]).strip():
            meta_lines.append(f"{f.replace('_', ' ').title()}: {row[f]}")
    meta_str = "\n".join(meta_lines)

    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = "<image>\n"
    if meta_str:
        prompt += f"Metadata:\n{meta_str}\n\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt


@torch.inference_mode()
def predict_mcq_index_metadata(row: pd.Series, image: Image.Image) -> int:
    return predict_mcq_index_with_prompt_builder(
        row,
        image,
        lambda r, include_answer=False: build_prompt_with_metadata(r, include_answer=include_answer),
    )


# Run-All friendly: keep demo off by default (it runs model inference).
RUN_SECTION14_DEMO = False
if RUN_SECTION14_DEMO:
    _r = val_df.iloc[0]
    _im = Image.open(IMG_DIR / _r["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    print("baseline:", predict_mcq_index(_r, _im))
    print("+metadata:", predict_mcq_index_metadata(_r, _im), "| gold:", int(_r["answer"]))
else:
    print("Section 14: demo skipped (RUN_SECTION14_DEMO=False).")


## 15. Your image is 224×224, zoom in on a few. If you can't read the axis labels or map text at that resolution, neither can your model. Try a bigger image size.

In [ ]:
# ── 15. Higher-resolution images for inference / scoring ────────────────────
# SmolVLM can use larger inputs; 224 may blur small text in diagrams.
# This does not change global `IMG_SIZE`; it only shows how to score at `IMG_SIZE_HIGH`.

IMG_SIZE_HIGH = 384  # try 384 or 448 if VRAM allows


def load_image_for_scoring(row: pd.Series, size: int) -> Image.Image:
    return Image.open(IMG_DIR / row["image_path"]).convert("RGB").resize((size, size), Image.BICUBIC)


@torch.inference_mode()
def predict_mcq_index_sized(row: pd.Series, size: int) -> int:
    """Same as Section 4 scorer, but image resized to `size`."""
    image = load_image_for_scoring(row, size)
    return predict_mcq_index(row, image)


# Run-All friendly: keep demo off by default (it runs extra inference at 2 resolutions).
RUN_SECTION15_DEMO = False
if RUN_SECTION15_DEMO:
    _r = val_df.iloc[0]
    print("224 pred:", predict_mcq_index_sized(_r, IMG_SIZE))
    print(f"{IMG_SIZE_HIGH} pred:", predict_mcq_index_sized(_r, IMG_SIZE_HIGH), "| gold:", int(_r["answer"]))
else:
    print("Section 15: demo skipped (RUN_SECTION15_DEMO=False).")


## 16. Data augmentation for images 
Random crop, slight rotation, brightness jitter. Helps the model generalize beyond the exact 224×224 framing.

In [ ]:
# ── 16. Light image augmentation (PIL-only, competition-safe) ─────────────
# Use only on **training** images. For MCQ scoring / test submission, keep eval deterministic (no aug).
import random
from PIL import ImageEnhance


def augment_train_image(img: Image.Image, p: float = 0.5) -> Image.Image:
    """Random brightness/contrast + small rotation. Keeps size unchanged."""
    img = img.copy()
    if random.random() < p:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.82, 1.18))
    if random.random() < p:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.88, 1.12))
    if random.random() < p:
        deg = random.uniform(-10.0, 10.0)
        img = img.rotate(deg, resample=Image.BICUBIC, expand=False, fillcolor=(255, 255, 255))
    return img


# Integration: in Section 8 `LoRAMCQDataset.__getitem__`, after loading `img`, add:
#   img = augment_train_image(img)   # only for training split
#
# Quick visual check (optional):
# from matplotlib import pyplot as plt
# raw = Image.open(IMG_DIR / train_df.iloc[0]["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
# plt.imshow(augment_train_image(raw)); plt.axis("off")


## 17. Gradient checkpointing 
increase batch size or image resolution without OOM, trading compute for memory.

In [ ]:
# ── 17. Gradient checkpointing (save VRAM during fine-tuning) ─────────────
# Trades extra compute for lower activation memory so you can use larger batch or resolution.
# Call on the **model you train** (e.g. `ft_model` from Section 8) after PEFT wrapping.


def enable_gradient_checkpointing(m) -> None:
    if hasattr(m, "config") and getattr(m.config, "use_cache", None) is not None:
        m.config.use_cache = False
    if hasattr(m, "enable_input_require_grads"):
        m.enable_input_require_grads()
    if hasattr(m, "gradient_checkpointing_enable"):
        m.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled on model.")
    else:
        print("No gradient_checkpointing_enable(); upgrade transformers or check model class.")


# Uncomment when running Section 8 fine-tuning:
# enable_gradient_checkpointing(ft_model)


## 18. Test results after training (val + saved submissions)

This section evaluates any saved adapters on a validation slice and writes:
- `data/val_eval_<run_name>_n<N>.csv`
- `data/val_summary.csv`

It does **not** overwrite `data/submission.csv`; each run’s submission stays as `data/submission_<run_name>.csv`.


In [ ]:
from pathlib import Path
from peft import PeftModel


def _predict_mcq_index_with(model_, processor_, row: pd.Series, image: Image.Image) -> int:
    """Same logic as `predict_mcq_index`, but explicit model/processor."""
    prompt = build_prompt(row, include_answer=False)

    n = int(row["num_choices"]) if "num_choices" in row else len(row["choices"])
    n = min(n, len(CHOICE_LETTERS))

    completions = [f" {CHOICE_LETTERS[i]}" for i in range(n)]
    full_texts = [prompt + c for c in completions]
    images = [image] * n

    inputs = processor_(text=full_texts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model_.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    inp_prompt = processor_(text=prompt, images=image, return_tensors="pt")
    prompt_len = int(inp_prompt["input_ids"].shape[1])

    outputs = model_(**inputs)
    logits = outputs.logits
    input_ids = inputs["input_ids"]
    attn_mask = inputs.get("attention_mask", None)

    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    log_probs = torch.log_softmax(shift_logits, dim=-1)
    token_logps = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)

    if attn_mask is not None:
        token_logps = token_logps * attn_mask[:, 1:].to(token_logps.dtype)

    start_pos = max(prompt_len - 1, 0)

    scores = []
    for i in range(n):
        inp_full = processor_(text=full_texts[i], images=image, return_tensors="pt")
        full_len = int(inp_full["input_ids"].shape[1])
        if full_len <= prompt_len:
            scores.append(float("-inf"))
            continue
        end_pos = full_len - 1
        slice_logps = token_logps[i, start_pos:end_pos]
        denom = slice_logps.numel() if slice_logps.numel() > 0 else 1
        scores.append((slice_logps.sum().item()) / denom)

    return int(np.argmax(scores))


@torch.inference_mode()
def evaluate_adapter_dir(
    adapter_dir: Path,
    *,
    run_name: str,
    val_n: int = 200,
) -> dict:
    """Load adapter onto the existing inference `model` and evaluate on `val_df` slice."""
    if not adapter_dir.exists():
        return {"run_name": run_name, "status": "missing", "adapter_dir": str(adapter_dir)}

    # Attach adapter on top of the already-loaded inference base model.
    m = PeftModel.from_pretrained(model, str(adapter_dir))

    subset = val_df.iloc[: min(val_n, len(val_df))].reset_index(drop=True)
    rows = []
    correct = 0
    for i in tqdm(range(len(subset)), desc=f"val {run_name}"):
        row = subset.iloc[i]
        img = Image.open(IMG_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        pred = _predict_mcq_index_with(m, processor, row, img)
        gold = int(row["answer"])
        ok = int(pred == gold)
        correct += ok
        rows.append(
            {
                "id": row["id"],
                "answer_true": gold,
                "answer_pred": int(pred),
                "correct": ok,
                "num_choices": int(row["num_choices"]),
                "image_path": row["image_path"],
            }
        )

    acc = correct / len(subset) if len(subset) else 0.0
    out_csv = DATA_DIR / f"val_eval_{run_name}_n{len(subset)}.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False)

    return {
        "run_name": run_name,
        "status": "ok",
        "adapter_dir": str(adapter_dir),
        "val_n": int(len(subset)),
        "val_acc": float(acc),
        "val_csv": str(out_csv),
    }


# --- What to evaluate (edit as needed) ---
RUNS_TO_CHECK = [
    (DATA_DIR / "dora_adapter", "section10_dora"),
    (DATA_DIR / "lora_s9_adapter", "section9_lora"),
    (DATA_DIR / "lora_s12_adapter", "section12_lora"),
    (DATA_DIR / "lora_adapter", "section8"),
]

VAL_CHECK_N = 200

summ = []
for adir, name in RUNS_TO_CHECK:
    summ.append(evaluate_adapter_dir(adir, run_name=name, val_n=VAL_CHECK_N))

summary_df = pd.DataFrame(summ)
summary_out = DATA_DIR / "val_summary.csv"
summary_df.to_csv(summary_out, index=False)
print("Saved summary to", summary_out.resolve())
display(summary_df.sort_values(["status", "val_acc"], ascending=[True, False]))
